In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import random
import os

# 1. Cố định Seed
def seed_everything(seed=2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(2026)

print("--- 🎯 BÀI TẬP 1: HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI (BINARY) ---")

# 2. Dữ liệu giả lập & Xử lý NaN bằng Median
X_raw = np.random.randn(200, 10)
nan_indices = np.random.choice(200, size=20, replace=False)
X_raw[nan_indices, 0] = np.nan

df = pd.DataFrame(X_raw, columns=[f'feat_{i}' for i in range(10)])
y = np.random.randint(0, 2, size=200)

df['feat_0'] = df['feat_0'].fillna(df['feat_0'].median())

# 3. Custom Dataset & DataLoader
class BinaryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

dataset = BinaryDataset(df, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# 4. Định nghĩa Mô hình
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = BinaryClassifier(input_dim=10)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 5. Training Loop
for epoch in range(1, 16):
    model.train()
    total_loss = 0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/15 | Average Loss: {total_loss/len(dataloader):.4f}")

--- 🎯 BÀI TẬP 1: HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI (BINARY) ---
Epoch 01/15 | Average Loss: 0.7002
Epoch 05/15 | Average Loss: 0.6573
Epoch 10/15 | Average Loss: 0.6451
Epoch 15/15 | Average Loss: 0.5826


In [2]:
print("--- 📈 BÀI TẬP 2: HUẤN LUYỆN MÔ HÌNH HỒI QUY (REGRESSION) ---")

X_reg = np.random.randn(150, 4)
y_reg = np.random.randn(150, 1)

class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

reg_loader = DataLoader(RegressionDataset(X_reg, y_reg), batch_size=16, shuffle=True)

class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.fc(x)

reg_model = Regressor()
reg_criterion = nn.MSELoss()
reg_optimizer = torch.optim.SGD(reg_model.parameters(), lr=0.01)

for epoch in range(1, 11):
    reg_model.train()
    total_loss = 0
    for batch_X, batch_y in reg_loader:
        reg_optimizer.zero_grad()
        preds = reg_model(batch_X)
        loss = reg_criterion(preds, batch_y)
        loss.backward()
        reg_optimizer.step()
        total_loss += loss.item()
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/10 | MSE Loss: {total_loss/len(reg_loader):.4f}")

--- 📈 BÀI TẬP 2: HUẤN LUYỆN MÔ HÌNH HỒI QUY (REGRESSION) ---
Epoch 01/10 | MSE Loss: 0.9231
Epoch 05/10 | MSE Loss: 0.8472
Epoch 10/10 | MSE Loss: 0.8219


In [3]:
print("--- 🚀 BÀI TẬP 3: CHẠY DỰ ĐOÁN TẬP TEST & XUẤT CSV ---")

test_ids = list(range(1001, 1051))
X_test_raw = np.random.randn(50, 10)
X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)

# Sử dụng mô hình `model` đã huấn luyện ở Cell 1
model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)
    predicted_labels = (outputs >= 0.5).int().squeeze().tolist()

def create_submission(test_ids, predictions, output_path='submission_ex3.csv'):
    df_sub = pd.DataFrame({
        'id': test_ids,
        'predict_label': predictions
    })
    df_sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ Đã xuất thành công file kết quả dự đoán tại: {output_path}")

create_submission(test_ids, predicted_labels)

--- 🚀 BÀI TẬP 3: CHẠY DỰ ĐOÁN TẬP TEST & XUẤT CSV ---
✅ Đã xuất thành công file kết quả dự đoán tại: submission_ex3.csv
